<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/01_Data_Scientist/advanced/06_model_interpretability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Interpretability — LIME, SHAP & Partial Dependence

Black-box models are powerful but opaque. Interpretability tools help you understand what your model learned, debug failures, satisfy regulatory requirements, and build trust.

**Topics:** Feature importance, partial dependence plots (PDP), LIME, SHAP values, global vs local explanations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# Load breast cancer dataset
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train GBM (complex, hard to interpret directly)
gbm = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
gbm.fit(X_train, y_train)
print(f'GBM test accuracy: {gbm.score(X_test, y_test):.4f}')

# Also train RF and LR for comparison
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

lr = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
lr.fit(X_train, y_train)
print(f'RF  test accuracy: {rf.score(X_test, y_test):.4f}')
print(f'LR  test accuracy: {lr.score(X_test, y_test):.4f}')

## 1. Feature Importance — Global Methods

In [ ]:
# Compare three types of global feature importance

# 1. Built-in impurity importance (RF/GBM)
builtin_imp = pd.Series(gbm.feature_importances_, index=feature_names).sort_values(ascending=True)

# 2. Permutation importance (model-agnostic, test set)
perm_result = permutation_importance(gbm, X_test, y_test, n_repeats=20, random_state=42)
perm_imp = pd.Series(perm_result.importances_mean, index=feature_names).sort_values(ascending=True)

# 3. Logistic regression coefficients (inherently interpretable)
lr_coef = pd.Series(np.abs(lr.named_steps['clf'].coef_[0]), index=feature_names).sort_values(ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 8))

builtin_imp.tail(15).plot(kind='barh', ax=axes[0], color='steelblue', alpha=0.8)
axes[0].set_title('GBM Built-in\nImpurity Importance')
axes[0].set_xlabel('Importance')

perm_imp.tail(15).plot(kind='barh', ax=axes[1], color='orange', alpha=0.8)
axes[1].set_title('Permutation Importance\n(GBM on test set)')
axes[1].set_xlabel('Mean accuracy decrease')

lr_coef.tail(15).plot(kind='barh', ax=axes[2], color='green', alpha=0.8)
axes[2].set_title('Logistic Regression\n|Coefficients| (standardized)')
axes[2].set_xlabel('|Coefficient|')

plt.suptitle('Global Feature Importance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('Top 5 features by permutation importance:')
for feat, val in perm_imp.tail(5).items():
    print(f'  {feat:<40} {val:.4f}')

## 2. Partial Dependence Plots (PDP)

In [ ]:
# PDP shows marginal effect of one (or two) feature(s) on predictions
# Averages out all other features: E[f(x1, X2, ...)] for varying x1

# Top features to investigate
top_features_idx = perm_imp.tail(4).index.tolist()
top_features_idx_num = [list(feature_names).index(f) for f in top_features_idx]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, (feat, feat_idx) in enumerate(zip(top_features_idx, top_features_idx_num)):
    # Manual PDP computation for clarity
    x_range = np.linspace(X_train[:, feat_idx].min(), X_train[:, feat_idx].max(), 100)
    X_copy = X_train.copy()
    pdp_vals = []
    for x_val in x_range:
        X_copy[:, feat_idx] = x_val
        pdp_vals.append(gbm.predict_proba(X_copy)[:, 1].mean())
    
    axes[i].plot(x_range, pdp_vals, 'steelblue', lw=2.5)
    
    # Rug plot: actual data distribution
    axes[i].scatter(X_train[:, feat_idx],
                    np.random.uniform(-0.02, 0.02, len(X_train)) + 0.02,
                    alpha=0.1, s=5, color='black')
    
    axes[i].set_xlabel(feat, fontsize=9)
    axes[i].set_ylabel('P(malignant)')
    axes[i].set_title(f'PDP: {feat}', fontsize=10)
    axes[i].axhline(0.5, color='red', ls='--', alpha=0.5, label='Decision boundary')
    axes[i].legend(fontsize=8)

plt.suptitle('Partial Dependence Plots — GBM on Breast Cancer', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# 2D interaction PDP
feat_a = top_features_idx_num[0]
feat_b = top_features_idx_num[1]
n_grid = 30
xa = np.linspace(X_train[:, feat_a].min(), X_train[:, feat_a].max(), n_grid)
xb = np.linspace(X_train[:, feat_b].min(), X_train[:, feat_b].max(), n_grid)
pdp_2d = np.zeros((n_grid, n_grid))

for i, va in enumerate(xa):
    for j, vb in enumerate(xb):
        X_copy = X_train.copy()
        X_copy[:, feat_a] = va
        X_copy[:, feat_b] = vb
        pdp_2d[j, i] = gbm.predict_proba(X_copy)[:, 1].mean()

plt.figure(figsize=(8, 6))
plt.contourf(xa, xb, pdp_2d, levels=20, cmap='RdYlGn')
plt.colorbar(label='P(malignant)')
plt.xlabel(top_features_idx[0]); plt.ylabel(top_features_idx[1])
plt.title('2D PDP — Feature Interaction')
plt.tight_layout(); plt.show()

## 3. LIME — Local Interpretable Model-agnostic Explanations

In [ ]:
# LIME: explain individual predictions by fitting a local linear model
# around the instance of interest (in perturbed feature space)

def lime_explain(model, X_instance, X_train, feature_names, n_samples=1000, kernel_width=0.75):
    """
    Manual LIME implementation for tabular data.
    1. Perturb X_instance by sampling around it
    2. Get model predictions on perturbed samples
    3. Weight by proximity (RBF kernel)
    4. Fit weighted linear model
    5. Return linear coefficients as explanation
    """
    n_features = len(X_instance)
    
    # Sample perturbations (Gaussian around instance, scaled to data range)
    perturbations = np.random.normal(0, 1, (n_samples, n_features))
    feature_std = X_train.std(axis=0) + 1e-8
    X_perturbed = X_instance + perturbations * feature_std * 0.5
    
    # Get model predictions
    preds = model.predict_proba(X_perturbed)[:, 1]
    
    # Compute weights: RBF kernel based on distance from instance
    distances = np.sqrt(np.sum((perturbations / kernel_width)**2, axis=1))
    weights = np.exp(-distances**2 / 2)
    
    # Fit weighted linear regression
    from sklearn.linear_model import Ridge
    lime_model = Ridge(alpha=1.0)
    lime_model.fit(X_perturbed, preds, sample_weight=weights)
    
    return lime_model.coef_

# Explain a specific prediction
instance_idx = 5  # test instance to explain
X_instance = X_test[instance_idx]
true_label = y_test[instance_idx]
pred_prob  = gbm.predict_proba(X_instance.reshape(1, -1))[0, 1]

print(f'Instance #{instance_idx}:')
print(f'  True label:   {"Malignant" if true_label==0 else "Benign"}')
print(f'  Predicted:    {"Malignant" if pred_prob<0.5 else "Benign"} (p={pred_prob:.3f})')

# Run LIME
lime_coefs = lime_explain(gbm, X_instance, X_train, feature_names)
lime_explanation = pd.Series(lime_coefs, index=feature_names)

# Visualize LIME explanation
lime_sorted = lime_explanation.sort_values()
colors = ['red' if v < 0 else 'green' for v in lime_sorted.values]

plt.figure(figsize=(10, 8))
bars = plt.barh(range(len(lime_sorted)), lime_sorted.values, color=colors, alpha=0.8)
plt.yticks(range(len(lime_sorted)), lime_sorted.index, fontsize=9)
plt.axvline(0, color='black', lw=0.8)
plt.xlabel('LIME coefficient (local linear approximation)')
plt.title(f'LIME Explanation — Instance #{instance_idx}\n'
          f'Pred: {"Malignant" if pred_prob<0.5 else "Benign"} (p={pred_prob:.3f}), '
          f'True: {"Malignant" if true_label==0 else "Benign"}')
plt.tight_layout(); plt.show()

print('\nTop 5 features driving this prediction:')
for feat, coef in lime_explanation.abs().sort_values(ascending=False).head(5).items():
    direction = '+' if lime_explanation[feat] > 0 else '-'
    print(f'  {direction} {feat:<40} |coef|={coef:.4f}')

## 4. SHAP Values

In [ ]:
# SHAP: Shapley values from cooperative game theory
# Each feature's contribution = weighted average of its marginal contribution
# across all possible feature subsets

# Properties:
# 1. Efficiency: SHAP values sum to f(x) - E[f(x)]
# 2. Symmetry: equal features get equal attribution
# 3. Dummy: zero-contribution features get SHAP=0
# 4. Additivity: consistent across model combinations

try:
    import shap
    print(f'SHAP {shap.__version__} available')
    SHAP_AVAILABLE = True
    
    # TreeExplainer: exact SHAP for tree models (fast!)
    explainer = shap.TreeExplainer(gbm)
    shap_values = explainer.shap_values(X_test)
    
    # Summary plot (global view of all features)
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False)
    plt.tight_layout(); plt.show()
    
    # Waterfall plot for one instance (local explanation)
    shap_exp = explainer(X_test)
    plt.figure(figsize=(10, 8))
    shap.plots.waterfall(shap_exp[instance_idx], show=False)
    plt.tight_layout(); plt.show()
    
    # Dependence plot: SHAP value vs feature value
    best_feat_idx = np.abs(shap_values).mean(0).argmax()
    plt.figure(figsize=(8, 5))
    shap.dependence_plot(best_feat_idx, shap_values, X_test,
                         feature_names=feature_names, show=False)
    plt.tight_layout(); plt.show()
    
except ImportError:
    SHAP_AVAILABLE = False
    print('SHAP not installed. Install: pip install shap')

# Manual SHAP for tiny model (illustrate concept)
print()
print('=== SHAP Concept: Shapley Values ===' )
print()
print('For a model with 2 features A and B:')
print('  f()      = baseline prediction (empty coalition)')
print('  f(A)     = prediction with only A')
print('  f(B)     = prediction with only B')
print('  f(A,B)   = full prediction')
print()
print('  SHAP(A) = 0.5*(f(A) - f()) + 0.5*(f(A,B) - f(B))')
print('  SHAP(B) = 0.5*(f(B) - f()) + 0.5*(f(A,B) - f(A))')
print()
print('  Property: SHAP(A) + SHAP(B) = f(A,B) - f() ✓')

# Toy example
f_empty = 0.10
f_A = 0.20
f_B = 0.25
f_AB = 0.50

shap_A = 0.5 * (f_A - f_empty) + 0.5 * (f_AB - f_B)
shap_B = 0.5 * (f_B - f_empty) + 0.5 * (f_AB - f_A)

print()
print('Toy example:')
print(f'  f()=0.10, f(A)=0.20, f(B)=0.25, f(A,B)=0.50')
print(f'  SHAP(A) = {shap_A:.3f}')
print(f'  SHAP(B) = {shap_B:.3f}')
print(f'  Sum = {shap_A + shap_B:.3f}  f(A,B) - f() = {f_AB - f_empty:.3f} ✓')

print()
print('When to use each method:')
print('  SHAP         — gold standard, consistent axioms, use for production models')
print('  LIME         — faster for large feature spaces, good for NLP/images')
print('  PDP          — understand feature-target relationship globally')
print('  Permutation  — simple, model-agnostic global importance')